# Week 9 — Restored stage (recovery)

Apply the three Week-2 classical enhancements (DCP for haze, bilateral-on-Y for JPEG, NL-Means+bilateral for noise) to all 720 distorted tiles, re-run YOLOv8s-OBB + HED + ORB, and compare distorted vs restored on each metric.

In [ ]:
from pathlib import Path
import pandas as pd
from src.figures import plot_distorted_vs_restored

REPO   = Path.cwd().parent
DSW    = REPO / "results" / "distortion_sweep"
RSW    = REPO / "results" / "restoration_sweep"
CLEAN  = REPO / "results" / "clean"
OUTFIG = REPO / "outputs" / "figures"
OUTFIG.mkdir(parents=True, exist_ok=True)

def agg(sweep, fname, col):
    df = pd.read_csv(sweep / fname, dtype={"level": str})
    return df.groupby(["distortion", "level", "snr_db_mean"], as_index=False)[col].mean()

clean_map = pd.read_csv(CLEAN / "perclass_detections.csv")["ap_iou50"].mean()
clean_f   = pd.read_csv(CLEAN / "edge_metrics.csv")["ods_f_score"].mean()
print(f"clean mAP@0.5 = {clean_map:.3f}   clean ODS F = {clean_f:.3f}")

## mAP@0.5 — distorted (red) vs restored (green)

In [ ]:
plot_distorted_vs_restored(
    agg(DSW, "perclass_detections.csv", "ap_iou50"),
    agg(RSW, "perclass_detections.csv", "ap_iou50"),
    value_col="ap_iou50",
    title="mAP@0.5: distorted vs restored",
    ylabel="mAP@0.5",
    out_path=OUTFIG / "recovery_map_vs_snr.png",
    clean_baseline=clean_map,
)

## HED edge F-score (ODS)

In [ ]:
plot_distorted_vs_restored(
    agg(DSW, "edge_metrics.csv", "ods_f_score"),
    agg(RSW, "edge_metrics.csv", "ods_f_score"),
    value_col="ods_f_score",
    title="Edge F-score (ODS): distorted vs restored",
    ylabel="ODS F-score",
    out_path=OUTFIG / "recovery_edgeF_vs_snr.png",
    clean_baseline=clean_f,
)

## ORB good-match ratio

In [ ]:
plot_distorted_vs_restored(
    agg(DSW, "orb_match.csv", "good_ratio"),
    agg(RSW, "orb_match.csv", "good_ratio"),
    value_col="good_ratio",
    title="ORB good-match ratio: distorted vs restored",
    ylabel="good_ratio",
    out_path=OUTFIG / "recovery_orb_vs_snr.png",
)

## Per-distortion recovery table (mean metric gain = restored − distorted)

In [ ]:
def gain(fname, col):
    d = pd.read_csv(DSW / fname, dtype={"level": str}).groupby("distortion")[col].mean()
    r = pd.read_csv(RSW / fname, dtype={"level": str}).groupby("distortion")[col].mean()
    return (r - d).round(3)

pd.DataFrame({
    "mAP@0.5":   gain("perclass_detections.csv", "ap_iou50"),
    "ODS F":     gain("edge_metrics.csv", "ods_f_score"),
    "ORB ratio": gain("orb_match.csv", "good_ratio"),
})